# Task3 Relaxation And Dephasing

这个 notebook 同样直接读取 4 个 config 文件：

- `required_tasks/task3_relaxation_dephasing.yaml`
- `templates/solvers/qutip.yaml`
- `templates/devices/single_qubit.yaml`
- `templates/pulses/single_qubit.yaml`

当前 QASM 解析器不支持 `delay[...]`，所以这里用重复的 `H - X - H` 块来拉长相干演化时间。块数取偶数时，理想输出仍然回到 `|0>`，便于直接读取错误率。

误差定义：

- `qubit_error_rate`：不加测量门时末态相对理想 `|0>` 的误差
- `readout_added_error_rate`：加上测量门后的总误差减去 `qubit_error_rate`，作为读出阶段附加误差 proxy


In [ ]:
from __future__ import annotations

import copy
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from qsim.workflow import run_task
from qsim.workflow.task_io import (
    load_device_config_file,
    load_pulse_config_file,
    load_solver_config_file,
    load_task_config_file,
)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'qsim').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root containing src/qsim')


ROOT = find_repo_root()
NOTEBOOK_DIR = ROOT / 'examples' / 'noise_simulation_tests'
TASK_PATH = NOTEBOOK_DIR / 'required_tasks' / 'task3_relaxation_dephasing.yaml'
SOLVER_PATH = ROOT / 'templates' / 'solvers' / 'qutip.yaml'
DEVICE_PATH = ROOT / 'templates' / 'devices' / 'single_qubit.yaml'
PULSE_PATH = ROOT / 'templates' / 'pulses' / 'single_qubit.yaml'

task_base = load_task_config_file(TASK_PATH)
solver_base = load_solver_config_file(SOLVER_PATH)
device_base = load_device_config_file(DEVICE_PATH)
pulse_base = load_pulse_config_file(PULSE_PATH)

ENGINE = 'qutip'
RAMSEY_BLOCKS = 4
MEASURE_DURATION_NS = 100.0
T1_SWEEP_US = [5.0, 8.0, 12.0, 20.0, 35.0, 60.0, 100.0, 160.0]

plt.style.use('seaborn-v0_8-whitegrid')
TASK_PATH, SOLVER_PATH, DEVICE_PATH, PULSE_PATH


In [ ]:
def build_task3_qasm(*, include_measure: bool, ramsey_blocks: int) -> str:
    qasm = [
        'OPENQASM 3;',
        'include "stdgates.inc";',
        'qubit[1] q;',
        'bit[1] c;',
    ]
    for _ in range(ramsey_blocks):
        qasm.extend(['h q[0];', 'x q[0];', 'h q[0];'])
    if include_measure:
        qasm.append('measure q[0] -> c[0];')
    return '\n'.join(qasm) + '\n'


def final_p1_from_result(result: dict) -> float:
    return float(result['results']['metrics']['population']['series']['1'][-1])


def error_rate_from_result(result: dict, *, target_p1: float) -> tuple[float, float]:
    final_p1 = final_p1_from_result(result)
    return abs(final_p1 - target_p1), final_p1


def run_task3_variant(*, t1_us: float, include_measure: bool) -> dict:
    task_cfg = copy.deepcopy(task_base)
    solver_cfg = copy.deepcopy(solver_base)
    device_cfg = copy.deepcopy(device_base)
    pulse_cfg = copy.deepcopy(pulse_base)

    t1_s = float(t1_us) * 1.0e-6
    t2_s = 0.6 * t1_s

    task_cfg.input.qasm_text = build_task3_qasm(include_measure=include_measure, ramsey_blocks=RAMSEY_BLOCKS)
    task_cfg.output.persist_artifacts = False
    task_cfg.output.export_plots = False
    task_cfg.output.export_dxf = False
    task_cfg.output.out_dir = str(
        (NOTEBOOK_DIR / 'runs' / 'notebooks' / 'task3_relaxation_dephasing' / f't1_{int(t1_us * 1000):08d}ns' / ('readout' if include_measure else 'memory')).resolve()
    )

    solver_cfg.run.engine = ENGINE
    solver_cfg.run.solver_mode = 'me'
    solver_cfg.run.seed = 12345
    solver_cfg.run.dt_s = 1.0e-9
    solver_cfg.run.t_end_s = 6.0e-7

    device_cfg.noise = {
        'model': 'markovian_lindblad',
        'T1_s': t1_s,
        'T2_s': t2_s,
    }
    pulse_cfg['gate_duration_ns'] = 20.0
    pulse_cfg['measure_duration_ns'] = MEASURE_DURATION_NS

    return run_task(task_cfg, solver_config=solver_cfg, device_config=device_cfg, pulse_config=pulse_cfg)


In [ ]:
rows = []
for t1_us in T1_SWEEP_US:
    memory_result = run_task3_variant(t1_us=t1_us, include_measure=False)
    readout_result = run_task3_variant(t1_us=t1_us, include_measure=True)

    qubit_error_rate, memory_final_p1 = error_rate_from_result(memory_result, target_p1=0.0)
    total_error_rate, readout_final_p1 = error_rate_from_result(readout_result, target_p1=0.0)
    readout_added_error_rate = max(total_error_rate - qubit_error_rate, 0.0)

    rows.append({
        't1_us': float(t1_us),
        't2_us': 0.6 * float(t1_us),
        'gamma1_per_us': 1.0 / float(t1_us),
        'memory_final_p1': memory_final_p1,
        'readout_final_p1': readout_final_p1,
        'qubit_error_rate': qubit_error_rate,
        'total_error_with_readout': total_error_rate,
        'readout_added_error_rate': readout_added_error_rate,
    })

df = pd.DataFrame(rows).sort_values('gamma1_per_us')
display(df)
df


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df['gamma1_per_us'], df['qubit_error_rate'], marker='o', linewidth=2, label='qubit_error_rate')
ax.plot(df['gamma1_per_us'], df['readout_added_error_rate'], marker='s', linewidth=2, label='readout_added_error_rate')
ax.set_xlabel('relaxation rate 1/T1 (1/us)')
ax.set_ylabel('error rate')
ax.set_title('Task3: relaxation/dephasing strength vs error rates')
ax.legend()
plt.tight_layout()
plt.show()
